In [1]:
"""
Constructs the meshes from the volumetric data and from the 2.5D shapes.
"""

import napari_spatialdata.constants.config
import spatialdata as sd
from pathlib import Path
from numpy.random import default_rng

from tissue_map_tools.igneous_converters import (  # noqa: F401
    from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes,
)
from tissue_map_tools.data_model.annotations_utils import (
    make_dtypes_compatible_with_precomputed_annotations,
)
import time  # noqa: F401
import shutil  # noqa: F401
from tissue_map_tools.converters import (  # noqa: F401
    from_spatialdata_points_to_precomputed_points,
)
from tissue_map_tools.data_model.annotations_utils import parse_annotations

RNG = default_rng(42)

SMALL_DATA = True

/Users/macbook/embl/projects/basel/3d-spatial-workshop-2025/.venv/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
out_path = Path.cwd() / "data"
sdata_zarr_path = out_path / "merfish_mouse_ileum.sdata.zarr"
precomputed_path = out_path / "merfish_mouse_ileum_precomputed" + ('' if SMALL_DATA else '_full')

# load the data
f = Path(sdata_zarr_path)
sdata = sd.read_zarr(f)

In [3]:
print(sd.get_extent(sdata["molecules"]))

{'x': (np.float64(112.0), np.float64(5720.0)), 'y': (np.float64(0.0), np.float64(9391.0)), 'z': (np.float64(0.0), np.float64(110.1455251))}


Let's subset the data in order to run this example notebook faster. Setting `SMALL_SDATA = False` will use the full data.

In [4]:
##
# subset the data
sdata_small = sd.bounding_box_query(
    sdata,
    axes=("x", "y", "z"),
    min_coordinate=[4000, 0, -10],
    max_coordinate=[5000, 1500, 200],
    target_coordinate_system="global",
)

/Users/macbook/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:909: UserWarning: The object has `points` element. Depending on the number of points, querying MAY suffer from performance issues. Please consider filtering the object before calling this function by calling the `subset()` method of `SpatialData`.
  return dispatch(args[0].__class__)(*args, **kw)


In [5]:
# we need to transform the vector data to match the image due to this issue:
# https://github.com/hms-dbmi/tissue-map-tools/issues/13
transformation = sd.transformations.get_transformation(sdata_small["stains"])
translation_vector = transformation.to_affine_matrix(
    input_axes=("x", "y", "z"), output_axes=("x", "y", "z")
)[:3, 3]
translation = sd.transformations.Translation(translation_vector, axes=("x", "y", "z"))
for _, element_name, _ in sdata_small.gen_spatial_elements():
    old_transformation = sd.transformations.get_transformation(
        sdata_small[element_name]
    )
    sequence = sd.transformations.Sequence([old_transformation, translation.inverse()])
    sd.transformations.set_transformation(
        sdata_small[element_name],
        transformation=sequence,
        to_coordinate_system="global",
    )
    if sd.models.get_model(sdata_small[element_name]) not in (
        sd.models.Image3DModel,
        sd.models.Labels3DModel,
    ):
        transformed = sd.transform(sdata_small[element_name], to_coordinate_system="global")
        sdata_small[element_name] = transformed

if SMALL_DATA:
    sdata = sdata_small

Let's convert the `SpatialData` Zarr storage to the [Neuroglancer Precomputed format](https://github.com/google/neuroglancer/blob/master/src/datasource/precomputed/annotations.md), to enable visualization with `neuroglancer`.

In [6]:
from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes(
    raster=sdata["dapi_labels"],
    precomputed_path=str(precomputed_path),
)

Converted OME-Zarr data to the Precomputed format (segmentation) at _full with pixel sizes {'x': 1000, 'y': 1000, 'z': 13768} and axes ['x', 'y', 'z'].
Volume Bounds:  Bbox([0, 0, 0],[5721, 9392, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[5721, 9392, 9], dtype=np.int32, unit='vx')


Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:05<00:00,  5.78s/it]


Volume Bounds:  Bbox([0, 0, 0],[2861, 4696, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[2861, 4696, 9], dtype=np.int32, unit='vx')


Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.57s/it]


Volume Bounds:  Bbox([0, 0, 0],[1431, 2348, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[1431, 2348, 9], dtype=np.int32, unit='vx')


Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.05it/s]


Volume Bounds:  Bbox([0, 0, 0],[716, 1174, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[716, 1174, 9], dtype=np.int32, unit='vx')


Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 2173.41it/s]

Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 1322.78it/s]

Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 2362.51it/s]

Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 3458.55it/s]

Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 3582.05it/s]

Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 3226.81it/s]

Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 48/48

We also create the meshes in the unsharded format (later we will use them in Vitessce, who only support the unsharded format so far).

In [8]:
from tissue_map_tools.igneous_converters import from_precomputed_raster_to_precomputed_meshes

from_precomputed_raster_to_precomputed_meshes(
    data_path=str(precomputed_path),
    mesh_name='mesh_mip_0_err_40_unsharded',
    sharded=False,
)

Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 999/999 [00:19<00:00, 51.41it/s]


Here we choose if subsetting the points data or we keep the full dataset, and we choose which features to keep.

In [9]:
subset = RNG.choice(len(sdata["molecule_baysor"]), 10000, replace=False)

print(sdata["molecule_baysor"].columns)
if SMALL_DATA:
    subset_df = sdata["molecule_baysor"].compute().iloc[subset]
else:
    subset_df = sdata["molecule_baysor"].compute()
subset_df = subset_df[
    [
        "x",
        "y",
        "z",
        "gene",
        "area",
        "mol_id",
        "x_raw",
        "y_raw",
        "z_raw",
        "brightness",
        "total_magnitude",
        "compartment",
        "nuclei_probs",
        "assignment_confidence",
        "cell",
        "is_noise",
        "layer",
    ]
]
subset_df

Index(['mol_id', 'x_raw', 'y_raw', 'z_raw', 'gene', 'area', 'brightness',
       'total_magnitude', 'qc_score', 'x', 'y', 'z', 'molecule_id',
       'confidence', 'compartment', 'nuclei_probs', 'cell',
       'assignment_confidence', 'is_noise', 'ncv_color', 'layer'],
      dtype='object')


,x,y,z,gene,area,mol_id,x_raw,y_raw,z_raw,brightness,total_magnitude,compartment,nuclei_probs,assignment_confidence,cell,is_noise,layer
0,1705.0,1271.0,0.000000,Maoa,4,3048145,-2935.386,-1218.580,2.5,2.021306,420.1126,Unknown,1.000000,0.625,75,False,1
1,1725.0,1922.0,0.000000,Maoa,4,3048147,-2933.229,-1147.614,2.5,1.828640,269.5874,Unknown,1.000000,0.950,189,False,1
2,1753.0,1863.0,0.000000,Maoa,5,3048148,-2930.104,-1154.062,2.5,2.001268,501.4615,Unknown,1.000000,1.000,188,False,1
3,1760.0,1865.0,0.000000,Maoa,7,3048149,-2929.339,-1153.784,2.5,1.960428,639.0364,Unknown,1.000000,1.000,188,False,1
4,1904.0,794.0,0.000000,Maoa,6,3048153,-2913.718,-1270.474,2.5,1.937280,519.3154,Unknown,1.000000,0.575,0,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819660,5704.0,38.0,13.768191,Hrh1,3,17236830,-2499.651,-1352.890,4.0,1.823321,199.7294,Cyto,0.533283,0.700,1020,False,2
819661,5685.0,43.0,13.768191,Htr4,3,17238042,-2501.752,-1352.326,4.0,1.833191,204.3206,Cyto,0.580681,0.650,1020,False,2
819662,5631.0,61.0,55.072763,Taar6,4,17239179,-2507.674,-1350.402,8.5,1.662809,184.0217,Unknown,0.668193,1.000,1002,False,5
819663,5720.0,62.0,0.000000,Taar7a,6,17239416,-2497.931,-1350.247,2.5,1.740916,330.4210,Cyto,0.657800,0.900,0,True,1


Ensure that the dtypes are compatible with the `neuroglancer` format. This will be made automatic in `tissue-map-tools`.

In [10]:
make_dtypes_compatible_with_precomputed_annotations(
    subset_df,
    max_categories=250,
    check_for_overflow=True,
)

,x,y,z,gene,area,mol_id,x_raw,y_raw,z_raw,brightness,total_magnitude,compartment,nuclei_probs,assignment_confidence,cell,is_noise,layer
0,1705.0,1271.0,0.000000,Maoa,4,3048145,-2935.385986,-1218.579956,2.5,2.021306,420.112610,Unknown,1.000000,0.625,75,0,1
1,1725.0,1922.0,0.000000,Maoa,4,3048147,-2933.229004,-1147.614014,2.5,1.828640,269.587402,Unknown,1.000000,0.950,189,0,1
2,1753.0,1863.0,0.000000,Maoa,5,3048148,-2930.104004,-1154.062012,2.5,2.001268,501.461487,Unknown,1.000000,1.000,188,0,1
3,1760.0,1865.0,0.000000,Maoa,7,3048149,-2929.339111,-1153.784058,2.5,1.960428,639.036377,Unknown,1.000000,1.000,188,0,1
4,1904.0,794.0,0.000000,Maoa,6,3048153,-2913.718018,-1270.473999,2.5,1.937280,519.315430,Unknown,1.000000,0.575,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819660,5704.0,38.0,13.768190,Hrh1,3,17236830,-2499.650879,-1352.890015,4.0,1.823321,199.729401,Cyto,0.533283,0.700,1020,0,2
819661,5685.0,43.0,13.768190,Htr4,3,17238042,-2501.751953,-1352.326050,4.0,1.833191,204.320602,Cyto,0.580681,0.650,1020,0,2
819662,5631.0,61.0,55.072762,Taar6,4,17239179,-2507.674072,-1350.401978,8.5,1.662809,184.021698,Unknown,0.668193,1.000,1002,0,5
819663,5720.0,62.0,0.000000,Taar7a,6,17239416,-2497.930908,-1350.246948,2.5,1.740916,330.420990,Cyto,0.657800,0.900,0,1,1


In [12]:
sdata["molecule_baysor"] = sd.models.PointsModel.parse(subset_df)

# raster data converted to precomputed expresses units in nm therefore let's multiply the points by 1000
# this will be made more ergonomic as part of the tissue-map-tools APIs
for ax in ["x", "y", "z"]:
    sdata["molecule_baysor"][ax] = sdata["molecule_baysor"][ax] * 1000

print("converting the points to the precomputed format")

start = time.time()
path = Path(precomputed_path) / "molecule_baysor"
if path.exists():
    shutil.rmtree(path)

converting the points to the precomputed format


In [13]:
from_spatialdata_points_to_precomputed_points(
    sdata["molecule_baysor"],
    precomputed_path=precomputed_path,
    points_name="molecule_baysor",
    limit=10000,
)
print(f"conversion of points: {time.time() - start}")

Processing grid level 0 with shape (1, 1, 1) and chunk size [5608000.        9391000.         110145.5234375]. Remaining points: 819665
Emitting 10000 points for grid cell (0, 0, 0)
Processing grid level 1 with shape (2, 2, 1) and chunk size [2804000.        4695500.         110145.5234375]. Remaining points: 809665
Emitting 10000 points for grid cell (0, 0, 0)
Emitting 10000 points for grid cell (0, 1, 0)
Emitting 10000 points for grid cell (1, 0, 0)
Emitting 10000 points for grid cell (1, 1, 0)
Processing grid level 2 with shape (4, 4, 1) and chunk size [1402000.        2347750.         110145.5234375]. Remaining points: 769665
Emitting 9220 points for grid cell (0, 0, 0)
Emitting 10000 points for grid cell (0, 1, 0)
Emitting 10000 points for grid cell (1, 0, 0)
Emitting 10000 points for grid cell (1, 1, 0)
Emitting 10000 points for grid cell (0, 2, 0)
Emitting 10000 points for grid cell (0, 3, 0)
Emitting 10000 points for grid cell (1, 2, 0)
Emitting 10000 points for grid cell (1, 3

In [14]:
print('done')

done


We have APIs to load the data from disk back to memory. Note the last columns `__spatial_index__` and `__chunk_key__`. These would enable compatibility with tools like [celldega](https://github.com/broadinstitute/celldega).

In [17]:
df_annotations = parse_annotations(Path(precomputed_path))
df_annotations

,x,y,z,gene,area,mol_id,x_raw,y_raw,z_raw,brightness,total_magnitude,compartment,nuclei_probs,assignment_confidence,cell,is_noise,layer,__spatial_index__,__chunk_key__
651794,2954000.0,2811000.0,96377.335938,Cps1,4,16702081,-2799.281006,-1050.797974,13.0,2.036481,435.052185,Unknown,0.964550,0.975,1108,0,8,spatial0,0_0_0
109770,3752000.0,6752000.0,27536.380859,Sdc1,5,3446830,-2712.323975,-621.414124,5.5,1.901822,398.834198,Cyto,0.009514,0.725,3338,0,3,spatial0,0_0_0
411491,4505000.0,976000.0,68840.953125,Maoa,7,9985399,-2630.342041,-1250.633057,10.0,1.758948,401.833313,Unknown,1.000000,0.575,890,0,6,spatial0,0_0_0
802835,4064000.0,7449000.0,13768.190430,Nlrp6,5,17113867,-2678.417969,-545.486389,4.0,1.770261,294.598602,Cyto,0.008998,1.000,3723,0,2,spatial0,0_0_0
396099,3558000.0,40000.0,0.000000,Clca3b,7,9877582,-2733.547119,-1352.660034,2.5,2.101502,884.300415,Unknown,0.997284,1.000,198,0,1,spatial0,0_0_0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367369,3337000.0,5082000.0,41304.574219,Slc5a1,8,9770339,-2757.532959,-803.313721,7.0,1.575751,301.190186,Cyto,0.010725,0.550,2258,0,4,spatial4,8_8_0
363023,3228000.0,4836000.0,68840.953125,Neat1,7,9763927,-2769.425049,-830.134521,10.0,1.969285,652.203186,Unknown,1.000000,0.525,2105,0,6,spatial4,8_8_0
356640,3156000.0,4997000.0,55072.761719,Net1,8,9754339,-2777.291992,-812.597412,8.5,1.870503,593.734924,Cyto,0.010267,1.000,2150,0,5,spatial4,8_8_0
349917,3106000.0,5121000.0,96377.335938,Slc12a2,4,9735836,-2782.708984,-799.095276,13.0,1.764960,232.820007,Unknown,0.875873,1.000,2230,0,8,spatial4,8_8_0
